In [1]:
### 시작 및 본인정보 반영
# 반드시 이 cell을 실행하시오
# 실행하지 않을 시 직접 하지 않은 것으로 간주
# 마지막 줄 출력으로 파일명을 설정하고 제출 필수
import platform, psutil, os, datetime
print(platform.processor())
print(str(round(psutil.virtual_memory().total / (1024.0 **3)))+"(GB)")
print(platform.system())
print(platform.version())
print(os.getcwd())
print(datetime.datetime.fromtimestamp(os.path.getctime(os.getcwd())))
print(datetime.datetime.fromtimestamp(os.path.getmtime(os.getcwd())))
print(datetime.datetime.fromtimestamp(os.path.getatime(os.getcwd())))
print(datetime.datetime.now())
title = 'DA_TimeSeries_Forecasting'    # 고정값
name = '김경원'    # 본인 이름을 작성
studentid = '20211011'    # 본인 학번을 작성
# 아래 강좌 명 중 본인이 수강하는 강과명 작성
# 비즈니스데이터사이언스이해, 디지털비즈니스애널리틱스, 비즈니스수요예측, 인공지능기반의사결정, 빅데이터 등
# 비즈니스혁신을위한데이터사이언스응용, 인공지능활용디지털경제플랫폼연구 등
course = '비즈니스수요예측'    
print('다음 출력을 파일명으로 설정하고 제출하시오:', name + '_' + studentid + '_' + course + '_' + title)

Intel64 Family 6 Model 186 Stepping 2, GenuineIntel
32(GB)
Windows
10.0.26100
C:\DataScience\Lecture\[DataScience]
2024-12-12 00:34:16.155373
2025-09-24 23:08:15.385646
2025-09-24 23:08:15.385646
2025-09-24 23:08:16.999110
다음 출력을 파일명으로 설정하고 제출하시오: 김경원_20211011_비즈니스수요예측_DA_TimeSeries_Forecasting


# **Import Library:** 분석에 사용할 모듈 설치

- 강의에서 배운 내용이든 아니든 `자유 설치 및 사용`

In [2]:
# Ignore the warnings
import warnings
# warnings.filterwarnings('always')
warnings.filterwarnings('ignore')

# System related and data input controls
import os

# Data manipulation and visualization
import pandas as pd
pd.options.display.float_format = '{:,.2f}'.format
pd.options.display.max_rows = 20
pd.options.display.max_columns = 20
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn import preprocessing

# Modeling algorithms
# General
import statsmodels.api as sm
from scipy import stats

# Regression
from scipy.stats import linregress
from sklearn.linear_model import LinearRegression, Ridge, RidgeCV, Lasso, LassoCV, ElasticNet, ElasticNetCV
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.kernel_ridge import KernelRidge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import LinearSVR, SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.tree import plot_tree, export_text, export_graphviz
from sklearn.ensemble import VotingRegressor, BaggingRegressor, RandomForestRegressor, AdaBoostRegressor, GradientBoostingRegressor, StackingRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from xgboost import plot_importance as plot_importance_xgb
from lightgbm import plot_importance as plot_importance_lgbm
from catboost import Pool, CatBoostRegressor
from mlxtend.regressor import StackingRegressor, StackingCVRegressor

# Model selection
from sklearn.model_selection import train_test_split, GridSearchCV

# Evaluation metrics
from sklearn import metrics
# for regression
from sklearn.metrics import mean_squared_error,  mean_absolute_error, mean_absolute_percentage_error

# **Data Loading:** 분석에 사용할 데이터 불러오기

<center><img src='Image/Advanced/Data_BikeSharingDemand.png' width='800'></center>

| **변수** | **설명** |
|:---:|:---:|
| **datetime** | 날짜시간정보 |
| **season** | 계절 |
| **holiday** | 휴일여부 |
| **workingday** | 주중/주말 |
| **weather** | 날씨 |
| **temp** | 온도 |
| **atemp** | 체감 온도 |
| **humidity** | 습도 |
| **windspeed** | 풍속도 |
| **casual** | 비회원 대여수량 |
| **registered** | 회원 대여수량 |
| **count** | 총 대여수량 |

In [3]:
location = os.path.join('.', 'Data', 'BikeSharingDemand', 'Bike_Sharing_Demand_Full.csv')
df = pd.read_csv(location)
df

,datetime,season,holiday,workingday,weather,temp,atemp,humidity,windspeed,casual,registered,count
0,2011-01-01 0:00,1,0,0,1,9.84,14.39,81,0.00,3,13,16
1,2011-01-01 1:00,1,0,0,1,9.02,13.63,80,0.00,8,32,40
2,2011-01-01 2:00,1,0,0,1,9.02,13.63,80,0.00,5,27,32
3,2011-01-01 3:00,1,0,0,1,9.84,14.39,75,0.00,3,10,13
4,2011-01-01 4:00,1,0,0,1,9.84,14.39,75,0.00,0,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...
17374,2012-12-31 19:00,1,0,1,2,10.66,12.88,60,11.00,11,108,119
17375,2012-12-31 20:00,1,0,1,2,10.66,12.88,60,11.00,8,81,89
17376,2012-12-31 21:00,1,0,1,1,10.66,12.88,60,11.00,7,83,90
17377,2012-12-31 22:00,1,0,1,1,10.66,13.63,56,9.00,13,48,61


# **Preprocessing:** 데이터 정리 및 패턴 추출하기

- 선택한 `데이터 및 비즈니스 목적` 달성을 위한 `알고리즘 기반 데이터분석`을 하기 위해 필요한 전처리 작업 `자유 진행`하고 `필요성 이유를 주석으로 작성`
- `전처리 순서는 변경가능`하지만 필요한 처리과정을 반드시 실행해야하며 `필요성 이유 미작성시 안한 것으로 간주`

- **Total:**
  
> (1) **[시간인덱스 적용 및 빈도설정]** 
> 
> (2) **[Down/Up Sampling]** 일별 데이터로 변환
>
> (3) **[Data Split]**
> - Y: `count`
> - X: Y를 제외한 나머지
> - Train: 2011년 1월 ~ 2012년 6월 (18개월)
> - Validate: 2012년 7월 ~ 2012년 9월 (3개월)
> - Test: 2012년 10월 ~ 2012년 12월 (3개월)
>
> `(1+2+3) 위 전처리 과정을 정리 후 함수로 작성 및 실행`
>
> (4) **[Train 시각화]**
> - 모든 독립변수들의 히스토그램(Histogram)
> - 모든 독립변수 별 종속변수의 분포 박스플랏(Boxplot): 필요에 따라 `시각화를 위한 독립변수의 구간화` 사용
> - 모든 독립변수와 종속변수의 상관관계 히드맵(Heatmap)

- **Train & Validate:**
  
> (5) **[시간변수 추출]** 연/분기/월/주/일/요일/휴일/근무일/공휴일여부/명절여부 (`미국 워싱턴 기준`)
>
> (6) **[시간특징 추출]** 추세/계절성/이동평균/차분/지연값
>
> (7) **[파생변수 및 나머지 전처리]** `Y 예측에 도움이 될 것으로 예상되는 자유로운 변수생성 및 전처리`
> - 필요여부를 판단하여 주석으로 작성하되, 자유롭고 창의적인 파생변수를 생성 + 알고리즘 학습 전 나머지 필요한 전처리 진행
>
> `(5+6+7) 위 전처리 과정을 정리 후 함수로 작성 및 실행`  

- **Test:**

> (8) **[Test 전처리]** 
> - Text 기간의 독립변수 값들은 미래가 도래해야만 생성되기 때문이 사용할 수가 없음
> - 자유롭고 창의적인 방법으로 Test 기간의 독립변수 값을 생성



## (1) **[시간인덱스 적용 및 빈도설정]** 

## (2) **[Down/Up Sampling]** 일별 데이터로 변환

## (3) **[Data Split]**
> - Y: `count`
> - X: Y를 제외한 나머지
> - Train: 2011년 1월 ~ 2012년 6월 (18개월)
> - Validate: 2012년 7월 ~ 2012년 9월 (3개월)
> - Test: 2012년 10월 ~ 2012년 12월 (3개월)

`(1+2+3) 위 전처리 과정을 정리 후 함수로 작성 및 실행`

## (4) **[Train 시각화]**
> - 모든 독립변수들의 히스토그램(Histogram)
> - 모든 독립변수 별 종속변수의 분포 박스플랏(Boxplot): 필요에 따라 `시각화를 위한 독립변수의 구간화` 사용
> - 모든 독립변수와 종속변수의 상관관계 히드맵(Heatmap)

## (5) **[시간변수 추출]** 연/분기/월/주/일/요일/휴일/근무일/공휴일여부/명절여부 (`미국 워싱턴 기준`)

## (6) **[시간특징 추출]** 추세/계절성/이동평균/차분/지연값

## (7) **[파생변수 및 나머지 전처리]** `Y 예측에 도움이 될 것으로 예상되는 자유로운 변수생성 및 전처리`
> - 필요여부를 판단하여 주석으로 작성하되, 자유롭고 창의적인 파생변수를 생성 + 알고리즘 학습 전 나머지 필요한 전처리 진행

`(5+6+7) 위 전처리 과정을 정리 후 함수로 작성 및 실행`  

## (8) **[Test 전처리]** 
> - Text 기간의 독립변수 값들은 미래가 도래해야만 생성되기 때문이 사용할 수가 없음
> - 자유롭고 창의적인 방법으로 Test 기간의 독립변수 값을 생성

# **Applying Algorithms:**

**(1)** `preprocessing_ME` 전처리 데이터에 `알고리즘` 사용하여 `학습(Train) 및 검증(Validate)` 예측 진행

- 사용 알고리즘은 10종: `Linear Regression, Ridge, Lasso, Elastic Net, Decision Tree, Random Forest, XGBoost, LightGBM, CatBoost, Stacking`
- 각 알고리즘은 필요에 따라 하리퍼파라미터 튜닝 진행
- 예측은 Train & Validate 모두에 대해 실행

---

**(2)** `Train & Validate 예측 성능을 검증지표` 표로 확인 후 `MSPE, MAPE, MedAPE 평균 기준 오름차순` 정렬

> (1) RMSE (Root Mean Squared Error)
> 
> (2) MSPE (Mean Squared Percentage Error)
> 
> (3) MAE (Mean Absolute Error)
> 
> (4) MAPE (Mean Absolute Percentage Error)
> 
> (5) MedAE (Median Absolute Error)
> 
> (6) MedAPE (Median Absolute Percentage Error)

---

**(3)** `Train & Validate` 데이터의 예측 결과를 `시각화로 표현`하여 얼마나 정확한지 확인

- 10개의 모델링 Train 정답 Y와 예측 Y를 1개의 시각화로 표현하고, 동일하게 Validate 에 대해서도 1개의 시각화로 표현
- 비교 성능이 검증지표와 시각적으로 유사하게 나타나는지, 그리고 어떤 모델링이 적합한지 주석으로 의견 작성

---

**(4)** 모델링 방향 정리

- `검증지표`를 기준으로 `Train & Validate` 각각에 대해 10가지 알고리즘 중 `어떤 알고리즘이 예측력이 높은지` 주석으로 작성
- 실제 `Test`에 활용하기 위해선 `Train & Validate` 중 `어떤 데이터의 예측력이 더욱 중요한지 + 최종 활용 알고리즘` 주석으로 작성

--- 

**(5)** 과적합 체크

- `Variance Inflation Factor(VIF)`를 사용하여 다중공선성 경향이 `낮은 변수부터 높은 변수까지 오름차순` DataFrame을 작성
- 독립변수를 다중공선성이 낮은변수부터 1개씩 추가하여 전체 변수를 다 넣을때까지 `MSPE, MAPE, MedAPE가 어떻게 변화`하는지 시각화
- Validate 기준 가장 성능이 높은 독립변수의 갯수와 이름을 주석으로 작성하고 `X_train, X_val를 해당 변수들로 필터링하여 이후 과정에 사용`
- `검증지표와 시각화`로 성능을 재확인

## **(1)** `preprocessing_ME` 전처리 데이터에 `알고리즘` 사용하여 `학습(Train) 및 검증(Validate)` 예측 진행

- 사용 알고리즘은 10종: `Linear Regression, Ridge, Lasso, Elastic Net, Decision Tree, Random Forest, XGBoost, LightGBM, CatBoost, Stacking`
- 각 알고리즘은 필요에 따라 하리퍼파라미터 튜닝 진행
- 예측은 Train & Validate 모두에 대해 실행

## **(2)** `Train & Validate 예측 성능을 검증지표` 표로 확인 후 `MSPE, MAPE, MedAPE 평균 기준 오름차순` 정렬

> (1) RMSE (Root Mean Squared Error)
> 
> (2) MSPE (Mean Squared Percentage Error)
> 
> (3) MAE (Mean Absolute Error)
> 
> (4) MAPE (Mean Absolute Percentage Error)
> 
> (5) MedAE (Median Absolute Error)
> 
> (6) MedAPE (Median Absolute Percentage Error)

## **(3)** `Train & Validate` 데이터의 예측 결과를 `시각화로 표현`하여 얼마나 정확한지 확인

- 10개의 모델링 Train 정답 Y와 예측 Y를 1개의 시각화로 표현하고, 동일하게 Validate 에 대해서도 1개의 시각화로 표현
- 비교 성능이 검증지표와 시각적으로 유사하게 나타나는지, 그리고 어떤 모델링이 적합한지 주석으로 의견 작성

## **(4)** 모델링 방향 정리

- `검증지표`를 기준으로 `Train & Validate` 각각에 대해 10가지 알고리즘 중 `어떤 알고리즘이 예측력이 높은지` 주석으로 작성
- 실제 `Test`에 활용하기 위해선 `Train & Validate` 중 `어떤 데이터의 예측력이 더욱 중요한지 + 최종 활용 알고리즘` 주석으로 작성

## **(5)** 과적합 체크

- 가장 Validate 성능이 높은 1개의 모델링에 대해서,
- `Variance Inflation Factor(VIF)`를 사용하여 다중공선성 경향이 `낮은 변수부터 높은 변수까지 오름차순` DataFrame을 작성
- 독립변수를 다중공선성이 낮은변수부터 1개씩 추가하여 전체 변수를 다 넣을때까지 `MSPE, MAPE, MedAPE가 어떻게 변화`하는지 시각화
- Validate 기준 가장 성능이 높은 독립변수의 갯수와 이름을 주석으로 작성하고 `X_train, X_val를 해당 변수들로 필터링 후 재학습 및 예측`
- `검증지표와 시각화`로 성능을 재확인

# **Explanation:** 실제 예측력이 높은 과거 이유를 설명하고 미래 설명 근거 제시

**(1) `Train Explanation:`** 과거 설명력

> - [SHAP Beeswarm Summary Plot](https://shap.readthedocs.io/en/latest/example_notebooks/api_examples/plots/beeswarm.html)을 사용하여,
> - Train 검증지표 성능이 가장 높았던 알고리즘의 `과거 종속변수에 영향을 주는 상위 10개의 변수 이름과 영향 방향` 설명력을 주석으로 작성

**(2) `Validate Explanation:`** 검증 설명력

> - [SHAP Beeswarm Summary Plot](https://shap.readthedocs.io/en/latest/example_notebooks/api_examples/plots/beeswarm.html)을 사용하여,
> - Validate 검증지표 성능이 가장 높았던 알고리즘의 `과거 종속변수에 영향을 주는 상위 10개의 변수 이름과 영향 방향` 설명력을 주석으로 작성

**(3) `설명력 비교 및 미래 예상:`**

> - `Train & Validate 설명력 상위 10개 변수 우선순위를 비교`하는 DataFrame을 만들고,
> - 향후(Test) 수요(종속변수)에 `어떻게 영향을 주게 될지 정량적` 근거를 기반으로 한 의견을 주석으로 작성
> - ex. 과거와 검증기간에 특정 변수가 종속변수에 영향을 주는 경향이 미래에는 종속변수에 블라블라 영향을 줄수 있으며 그 근거는 블라블라~


## **XAI(eXplainable AI):** 설명가능한 인공지능

**1) SHAP(SHapley Additive exPlanations) 알고리즘 배경:** 설명 가능한 AI를 위한 방법론

- **AI 블랙박스 이슈:** 많은 AI 모델(특히 딥러닝, 랜덤 포레스트, XGBoost 등)은 예측 성능이 뛰어나지만, 내부 동작을 이해하거나 해석하기 어려운 `블랙박스 모델로 간주`

<center><img src='Image/Advanced/Blackbox_vs_Whitebox.webp' width='600'>(https://liquidity-provider.com/articles/demystifying-black-box-ai-and-its-use-cases/)</center>

> - 모델들이 어떻게 결정을 내리는지 이해하지 못하면, `AI의 신뢰성이나 공정성을 평가하기 어려움`
> - 특히 `의료, 금융, 법률 등 중요한 분야일수록 그 문제가 더욱 중요`하며 해결 필요성이 높음
> - SHAP은 `게임 이론에서 유래한 Shapley 값을 기반`으로 `각 변수가 모델의 예측에 미친 영향을 정량적으로 설명`
> - SHAP는 `2017년에 Scott Lundberg & Su-In Lee에 의해 소개`

---

**2) 필요성:**

<center><img src='Image/Advanced/Blackbox_vs_Whitebox_Example.webp' width='600'>(https://www.analyticsvidhya.com/blog/2019/08/decoding-black-box-step-by-step-guide-interpretable-machine-learning-models-python/)</center>

**(1) 모델 해석 가능성 제공:** 예측을 할 때 `각 변수(Feature)가 얼마나 영향력이 큰지를 시각화`하여 결정 과정을 명확하게 설명

**(2) 모델 공정성 검토:** 특정 특성(예: 성별, 나이 등)이 얼마나 예측에 영향을 미쳤는지 분석 함으로써 `모델의 편향과 공정성 여부를 파악`

**(3) 신뢰성 향상:** 사용자는 모델의 예측이 `왜 그런 결과를 도출했는지 이해할 수 있기에 신뢰도가 높아짐`

---

**3) 특징 및 세부기능:**

- **특징**

> - Shapley 값은 게임 이론에서 유래된 개념으로, `각 참여자가 게임에서 얻은 총 이익을 공정하게 분배하는 방법`
> - AI에서 Shapley 값은 `각 변수가 모델 예측에 기여하는 정도`를 정의하는 데 사용
> - Shapley 값은 `모든 가능한 변수 조합에 대해 각 변수가 예측에 미친 영향을 계산`
> - **Local(Individual) Explanation:** 변수 뿐만 아니라 `각 고객별(Sample) 변수들의 기여도로 개별 예측 설명` 유용
> - **Global(Total) Explanation:** 고객별 개별 예측 설명을 합산하여 `전체 데이터셋에서 변수들의 모델 예측 영향을 설명`

- **세부기능:**

> **(1) Decison & Force Plot:** `개별 예측`을 이끌어낸 `변수들의 SHAP 기여도를 직관적 시각화`
>
> <center><img src='Image/Advanced/SHAP_Decisionplot.webp' width='600'></center>
> <center><img src='Image/Advanced/SHAP_Text.png' width='600'></center>
>
> **(2) Dependence Plot:** `전체 예측`을 이끌어낸 `각 변수의 실제 값과 SHAP 예측치의 관계`를 시각화
> 
> <center><img src='Image/Advanced/SHAP_Dependenceplot.webp' width='600'></center>
>
> **(3) Summary Plot:** 각 변수가 모델 `전체 예측`에 미친 기여 영향력을 한눈에 분포로 시각화`
>
> <center><img src='Image/Advanced/SHAP_Heatmap.webp' width='600'></center>
> <center><img src='Image/Advanced/SHAP_Beeswarm.webp' width='600'>(https://shap.readthedocs.io/en/latest/index.html)</center>

| **구분** | **Decision Plot** | **Summary Plot** |
|:---:|:---:|:---:|
| **대상** | **개별 샘플** | **전체 데이터** |
| **해석방법** | **> X축:** 변수의 예측 기여 SHAP 값<br>**> Y축:** 변수 중요도 내림차순 기반 누적 기여 SHAP값<br>**> 선그래프:** 기울기 기반 해당 변수 영향력 <br>(기울기 변화가 클수록 예측에 크게 기여하기에 위로 갈수록 중요) | **> X축:** 변수의 예측 기여 SHAP 값<br>**> Y축:** 변수 중요도 내림차순 기반 변수별 샘플기여 누적 SHAP 값<br>**> 분포선그래프:** 변수 값의 변화에 따른 예측 기여 방향과 정도<br>- **파란색:** 변수의 낮은값에 대한 SHAP 값 분포<br>- **빨간색:** 변수의 높은값에 대한 SHAP 값 분포<br>- **파란색->빨간색 분포변화:** 예측에 대한 긍정기여<br>- **빨간색->파란색 분포변화:** 예측에 대한 부정기여<br>(분포 변화가 클수록 예측에 크게 기여하기에 위로 갈수록 중요 변수) |
| **설명** | 개별 예측의 결정 경로를 누적 SHAP 값으로 시각화<br>(선 = 개별기여, 변수 = 결정경로 우선순위) | 개별 예측의 변수별 기여방향을 누적하여 SHAP 값 분포로 시각화<br>(점 = 개별기여, 색 = 기여방향, 변수 = 전체결정 우선순위)  |
| **활용목적** | 개별 예측의 주요변수 식별과 결정과정 추적<br>샘플 결과간 비교를 통한 개별 샘플의 인싸이트 확보 | 전체 예측의 주요변수 식별과 영향방향 해석<br>모델의 전반적인 이해로 의사결정 지원 |


---

**4) 요약:**

- SHAP는 `AI 모델의 예측을 투명하게 설명하는 현존하는 강력한 도구`
- SHAP 값을 통해 모델의 `예측을 직관적으로 이해하고, 모델의 공정성, 신뢰성, 투명성을 개선`
- SHAP는 설명 가능한 AI의 핵심 기술로 자리잡고 있으며, `다양한 분야에서 실용적으로 활용`

## **(1) `Train Explanation:`** 과거 설명력

> - [SHAP Beeswarm Summary Plot](https://shap.readthedocs.io/en/latest/example_notebooks/api_examples/plots/beeswarm.html)을 사용하여,
> - Train 검증지표 성능이 가장 높았던 알고리즘의 `과거 종속변수에 영향을 주는 상위 10개의 변수 이름과 영향 방향` 설명력을 주석으로 작성

## **(2) `Validate Explanation:`** 검증 설명력

> - [SHAP Beeswarm Summary Plot](https://shap.readthedocs.io/en/latest/example_notebooks/api_examples/plots/beeswarm.html)을 사용하여,
> - Validate 검증지표 성능이 가장 높았던 알고리즘의 `과거 종속변수에 영향을 주는 상위 10개의 변수 이름과 영향 방향` 설명력을 주석으로 작성

## **(3) `설명력 비교 및 미래 예상:`**

> - `Train & Validate 설명력 상위 10개 변수 우선순위를 비교`하는 DataFrame을 만들고,
> - 향후(Test) 수요(종속변수)에 `어떻게 영향을 주게 될지 정량적` 근거를 기반으로 한 의견을 주석으로 작성
> - ex. 과거와 검증기간에 특정 변수가 종속변수에 영향을 주는 경향이 미래에는 종속변수에 블라블라 영향을 줄수 있으며 그 근거는 블라블라~

# **Forecasting:** 실제 미래의 수요를 예측

**(1)** 과거를 학습하여 검증기간의 성능 상위 1위와 비교를 위해 2위 및 3위의 알고리즘으로 실제 미래 3개월(Test)을 예측해보고,

**(2)** 얼마나 현실성이 있는 예측이라고 생각하는지 주석으로 작성하고 미래의 수요를 기반으로 한 비즈니스 전략을 주석으로 작성


## **(1)** 과거를 학습하여 검증기간의 성능 상위 1위와 비교를 위해 2위 및 3위의 알고리즘으로 실제 미래 3개월(Test)을 예측해보고,

## **(2)** 얼마나 현실성이 있는 예측이라고 생각하는지 주석으로 작성하고 미래의 수요를 기반으로 한 비즈니스 전략을 주석으로 작성

In [95]:
### 시작 및 본인정보 반영
# 반드시 이 cell을 실행하시오
# 실행하지 않을 시 직접 하지 않은 것으로 간주
# 마지막 줄 출력으로 파일명을 설정하고 제출 필수
import platform, psutil, os, datetime
print(platform.processor())
print(str(round(psutil.virtual_memory().total / (1024.0 **3)))+"(GB)")
print(platform.system())
print(platform.version())
print(os.getcwd())
print(datetime.datetime.fromtimestamp(os.path.getctime(os.getcwd())))
print(datetime.datetime.fromtimestamp(os.path.getmtime(os.getcwd())))
print(datetime.datetime.fromtimestamp(os.path.getatime(os.getcwd())))
print(datetime.datetime.now())
title = 'DA_TimeSeries_Forecasting'    # 고정값
name = '김경원'    # 본인 이름을 작성
studentid = '20211011'    # 본인 학번을 작성
# 아래 강좌 명 중 본인이 수강하는 강과명 작성
# 비즈니스데이터사이언스이해, 디지털비즈니스애널리틱스, 비즈니스수요예측, 인공지능기반의사결정, 빅데이터 등
# 비즈니스혁신을위한데이터사이언스응용, 인공지능활용디지털경제플랫폼연구 등
course = '비즈니스수요예측'    
print('다음 출력을 파일명으로 설정하고 제출하시오:', name + '_' + studentid + '_' + course + '_' + title)

Intel64 Family 6 Model 186 Stepping 2, GenuineIntel
32(GB)
Windows
10.0.26100
C:\DataScience\Lecture\[DataScience]
2024-12-12 00:34:16.155373
2025-09-23 01:43:33.434662
2025-09-23 01:43:33.434662
2025-09-23 01:43:53.427206
다음 출력을 파일명으로 설정하고 제출하시오: 김경원_20211011_비즈니스수요예측_DA_TimeSeries_Forecasting
